# IOAI — 2025 At Home Round Weather — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [103]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "At-Home-Round/Weather"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

작업 폴더: /content/IOAI-2025/At-Home-Round/Weather
내용: ['Weather.ipynb', 'Weather_Solution.ipynb', 'best_model.pth', 'figs', 'metric_v5', 'pred_a.npz', 'submission.zip', 'wcache']


<img src="https://github.com/scvcoder/ioai-colab/blob/main/notebooks/figs/IOAI-Logo.png?raw=1" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), At-Home Round](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/At-Home-Round/Weather/Weather_Solution.ipynb)

In [104]:
import numpy as np
import torch
import math
import random
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path
import zipfile
from tqdm import tqdm
from datetime import datetime, timedelta
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# --- 재현성(원본엔 없던 시드 고정) ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TEST_PATH = "wcache/"
os.makedirs(TEST_PATH, exist_ok=True)
import urllib.request as _u
for _f in ["dataset.npz", "metadata.csv"]:
    _p = TEST_PATH + _f
    if not os.path.exists(_p):
        print("원격 미러에서 다운로드:", _f)
        _u.urlretrieve(
            "https://huggingface.co/datasets/Silicon23/ioai2025-athome-satellite-images/resolve/main/data/"
            + _f + "?download=true", _p)
DATASET_PATH  = TEST_PATH + "dataset.npz"
METADATA_PATH = TEST_PATH + "metadata.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 이후 전부 이 값 사용(하드코딩 cuda 제거)
BASELINE_MODEL = TEST_PATH + "model_weights.pth"
print("DEVICE =", DEVICE)

DEVICE = cuda


In [ ]:
# Load from .npz file
loaded = np.load(DATASET_PATH)
X_train_128 = loaded['X_train_128']
X_train_256 = loaded['X_train_256']
Y_train_128 = loaded['Y_train_128']
Y_train_256 = loaded['Y_train_256']
X_test_128 = loaded['X_test_128']
X_test_256 = loaded['X_test_256']
Y_test_128 = loaded['Y_test_128']
Y_test_256 = loaded['Y_test_256']

In [ ]:
df = pd.read_csv(METADATA_PATH)
df_train_128 = df[(df['size'] == 128) & (df['split'] == 'train')][['i', 'j', 'start_time']]
df_train_256 = df[(df['size'] == 256) & (df['split'] == 'train')][['i', 'j', 'start_time']]
df_test_128 = df[(df['size'] == 128) & (df['split'] == 'test')][['i', 'j', 'start_time']]
df_test_256 = df[(df['size'] == 256) & (df['split'] == 'test')][['i', 'j', 'start_time']]

In [ ]:
def pixel_to_latlon(i, j):
    """
    Converts pixel indices (row i, column j) to geographic latitude and longitude.

    Args:
        i (int): Row index (from 0 at the top to 3499 at the bottom).
        j (int): Column index (from 0 at the left to 6999 at the right).

    Returns:
        tuple: (latitude, longitude) as floats.

    This function is used to convert from image/pixel coordinates
    (such as a satellite image) to actual map coordinates.
    """
    lat = 60.0 - i * 0.01   # Each row down is 0.01 degree further south
    lon = -130.0 + j * 0.01 # Each column right is 0.01 degree further east
    return lat, lon


def solar_elevation(x, y, dt_utc):
    """
    Calculates the sun's elevation angle above the horizon for a specific
    location (pixel) and UTC time.

    Args:
        x (int): Row index in the image (vertical position).
        y (int): Column index in the image (horizontal position).
        dt_utc (datetime or str): The date and time in UTC (string or datetime).

    Returns:
        float: Solar elevation angle in degrees.

    This function tells you "how high is the sun in the sky"
    for a given place and time.
    """
    # If time is given as string, convert to datetime
    if isinstance(dt_utc, str):
        dt_utc = datetime.strptime(dt_utc, '%Y-%m-%d %H:%M:%S.%f')

    # Convert pixel indices to latitude and longitude
    lat, lon = pixel_to_latlon(x, y)

    # Estimate local time (in hours) by longitude (15 degrees = 1 hour)
    timezone_offset = lon / 15.0
    local_time = dt_utc.hour + dt_utc.minute / 60 + timezone_offset

    # Day of year (1-365/366)
    N = dt_utc.timetuple().tm_yday

    # Solar declination: angle between sun's rays and Earth's equator
    decl = 23.44 * math.sin(math.radians(360 / 365 * (N - 81)))

    # Hour angle: how far in time from solar noon
    H = 15 * (local_time - 12)  # degrees

    # Convert everything to radians for math functions
    phi = math.radians(lat)
    delta = math.radians(decl)
    H = math.radians(H)

    # Calculate elevation using spherical trigonometry
    sin_h = math.sin(phi) * math.sin(delta) + math.cos(phi) * math.cos(delta) * math.cos(H)
    return sin_h


def parse_goes_time(timestr):

    """
    Converts a GOES satellite timestamp string into a Python datetime.

    Args:
        timestr (str): Timestamp string, e.g. 's20242891100205'.

    Returns:
        datetime: The parsed datetime.

    Format explanation:
        - 's20242891100205' means:
            year = 2024,
            day-of-year = 289,
            hour = 11,
            minute = 00,
            second = 20,
            tenths of a second = 5
    """
    timestr = str(timestr)
    if not (timestr[:1] == "s" and timestr[1:5].isdigit()):
        from datetime import datetime as _dt
        return _dt.fromisoformat(timestr)
    year = int(timestr[1:5])
    doy = int(timestr[5:8])
    hour = int(timestr[8:10])
    minute = int(timestr[10:12])
    second = int(timestr[12:14])
    micro = int(timestr[14]) * 100000  # tenths of a second
    return datetime(year, 1, 1) + timedelta(days=doy - 1, hours=hour, minutes=minute, seconds=second, microseconds=micro)


def df_to_list_solar_elevation(df):
    hs = []
    for i, j, t in zip(df['i'], df['j'], df['start_time']):
        h = solar_elevation(i, j, t)
        dt = parse_goes_time(t)

        # 正弦编码月份 (1-12 -> 0-2π)
        month_sin = math.sin(2 * math.pi * (dt.month - 1) / 12)
        month_cos = math.cos(2 * math.pi * (dt.month - 1) / 12)

        # 正弦编码一天内的时间 (小时 -> 0-2π)
        hour_decimal = dt.hour + dt.minute / 60 + dt.second / 3600
        time_sin = math.sin(2 * math.pi * hour_decimal / 24)
        time_cos = math.cos(2 * math.pi * hour_decimal / 24)

        # 正弦编码经纬度 (假设i和j是经纬度坐标)
        # 假设i是纬度，j是经度，需要根据实际数据范围调整
        i, j = pixel_to_latlon(i, j)
        lat_sin = math.sin(2 * math.pi * i / 180)  # 纬度范围 -90到90
        lat_cos = math.cos(2 * math.pi * i / 180)
        lon_sin = math.sin(2 * math.pi * j / 360)  # 经度范围 -180到180
        lon_cos = math.cos(2 * math.pi * j / 360)

        # 组合所有编码特征
        encoded_features = [h, month_sin, month_cos, time_sin, time_cos,
                          lat_sin, lat_cos, lon_sin, lon_cos]
        hs.append(encoded_features)
    return hs


In [ ]:
feats_train_128 = df_to_list_solar_elevation(df_train_128)
feats_train_256 = df_to_list_solar_elevation(df_train_256)
feats_test_128  = df_to_list_solar_elevation(df_test_128)
feats_test_256  = df_to_list_solar_elevation(df_test_256)
COND_DIM = 9
print("조건 벡터 차원:", COND_DIM)

In [ ]:
class SolarDataset(Dataset):
    """단일 해상도. 채널별(샘플 내) 표준화 후 (x, y, cond[9]) 반환. 불필요한 업/다운샘플 없음."""
    def __init__(self, X, Y, feats):
        self.X, self.Y, self.feats = X, Y, feats
    def __len__(self):
        return len(self.X)
    @staticmethod
    def _norm(x):                       # x: [C,H,W]
        mean = x.mean(dim=(1, 2), keepdim=True)
        std  = x.std(dim=(1, 2), keepdim=True)
        return (x - mean) / (std + 1e-6)
    def __getitem__(self, idx):
        x = self._norm(torch.as_tensor(self.X[idx], dtype=torch.float32))
        y = torch.as_tensor(self.Y[idx], dtype=torch.float32).unsqueeze(0)
        cond = torch.as_tensor(self.feats[idx], dtype=torch.float32)   # 9개 전부
        return x, y, cond

def split_idx(n, val_frac=0.15, seed=SEED):
    perm = np.random.default_rng(seed).permutation(n)
    n_val = max(1, int(n * val_frac))
    return perm[n_val:], perm[:n_val]

tr128, va128 = split_idx(len(X_train_128))
tr256, va256 = split_idx(len(X_train_256))

def take(lst, idx):                     # 파이썬 리스트 인덱싱 헬퍼
    return [lst[k] for k in idx]

train_ds_128 = SolarDataset(X_train_128[tr128], Y_train_128[tr128], take(feats_train_128, tr128))
val_ds_128   = SolarDataset(X_train_128[va128], Y_train_128[va128], take(feats_train_128, va128))
train_ds_256 = SolarDataset(X_train_256[tr256], Y_train_256[tr256], take(feats_train_256, tr256))
val_ds_256   = SolarDataset(X_train_256[va256], Y_train_256[va256], take(feats_train_256, va256))
# 공개 테스트셋(채점 프록시) — 원본 순서 보존
test_ds_128 = SolarDataset(X_test_128, Y_test_128, feats_test_128)
test_ds_256 = SolarDataset(X_test_256, Y_test_256, feats_test_256)

# 해상도별 로더(각 배치는 단일 해상도). BatchNorm size-1 방지로 train 은 drop_last=True
train_loaders = [DataLoader(train_ds_128, batch_size=32, shuffle=True, drop_last=True),
                 DataLoader(train_ds_256, batch_size=16, shuffle=True, drop_last=True)]
val_loaders   = [DataLoader(val_ds_128, batch_size=32), DataLoader(val_ds_256, batch_size=16)]
test_loaders  = [DataLoader(test_ds_128, batch_size=32), DataLoader(test_ds_256, batch_size=16)]

def iterate_mixed(loaders):
    """128/256 배치를 무작위로 섞어 내보냄(각 배치는 단일 해상도라 안전)."""
    iters = [iter(l) for l in loaders]
    pending = list(range(len(iters)))
    while pending:
        k = random.choice(pending)
        try:
            yield next(iters[k])
        except StopIteration:
            pending.remove(k)

print(f"train 128/256 = {len(train_ds_128)}/{len(train_ds_256)} | "
      f"val = {len(val_ds_128)}/{len(val_ds_256)} | test = {len(test_ds_128)}/{len(test_ds_256)}")

In [ ]:
class FiLM(nn.Module):
    """조건벡터 -> (gamma, beta) 로 bottleneck 변조. (1+gamma) 라 초기엔 항등에 가깝다."""
    def __init__(self, cond_dim, n_ch, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * n_ch))
    def forward(self, m, cond):
        gamma, beta = self.net(cond).chunk(2, dim=1)
        gamma = gamma.view(m.size(0), m.size(1), 1, 1)
        beta  = beta.view(m.size(0), m.size(1), 1, 1)
        return m * (1 + gamma) + beta

class UNetSegmentation(nn.Module):
    def __init__(self, in_channels, cond_dim=COND_DIM):
        super().__init__()
        self.encoder1 = self.conv_block(in_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.mid = self.conv_block(512, 1024)
        self.film = FiLM(cond_dim, 1024)                  # ← 조건 주입
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2); self.dec4 = self.conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2);  self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2);  self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2);   self.dec1 = self.conv_block(128, 64)
        self.final = nn.Conv2d(64, 1, kernel_size=1)
    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x, cond):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        m  = self.mid(self.pool(e4))
        m  = self.film(m, cond)                           # ← 9차원 조건 전체 사용
        d4 = self.dec4(torch.cat([self.up4(m), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, target, eps=1e-6):
    pred = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - (2 * inter + eps) / (union + eps)).mean()

def seg_loss(logits, y):
    logits = torch.clamp(logits, -20, 20)     # 안정화
    return bce_loss(logits, y) + dice_loss(logits, y)

In [ ]:
@torch.no_grad()
def evaluate(model, loaders, device=DEVICE, threshold=0.5):
    model.eval(); model.to(device)
    iou_sum = dice_sum = 0.0; n = 0
    TP = FP = FN = 0.0
    y_true, y_pred = [], []
    for loader in loaders:
        for x, y, cond in loader:
            x, y, cond = x.to(device), y.to(device), cond.to(device)
            preds = (torch.sigmoid(model(x, cond)) > threshold).float()
            inter = (preds * y).sum(dim=(1, 2, 3))
            union = ((preds + y) > 0).float().sum(dim=(1, 2, 3))
            psum, ysum = preds.sum(dim=(1, 2, 3)), y.sum(dim=(1, 2, 3))
            # 정답·예측 둘 다 빈 마스크면 완벽 → 1 로 처리(원본은 0 으로 깎였음)
            iou  = torch.where(union == 0, torch.ones_like(union), inter / (union + 1e-6))
            dice = torch.where((psum + ysum) == 0, torch.ones_like(psum), 2 * inter / (psum + ysum + 1e-6))
            iou_sum += iou.sum().item(); dice_sum += dice.sum().item(); n += x.size(0)
            TP += (preds * y).sum().item(); FP += (preds * (1 - y)).sum().item(); FN += ((1 - preds) * y).sum().item()
            y_true.append((ysum > 0).cpu()); y_pred.append((psum > 0).cpu())
    y_true = torch.cat(y_true).float(); y_pred = torch.cat(y_pred).float()
    acc = (y_true == y_pred).float().mean().item()
    dice_final, iou_final = dice_sum / n, iou_sum / n
    prec, rec = TP / (TP + FP + 1e-6), TP / (TP + FN + 1e-6)
    score = (dice_final + acc) / 2
    print(f"IoU {iou_final:.4f} | Dice {dice_final:.4f} | Prec {prec:.4f} | "
          f"Recall {rec:.4f} | RainAcc {acc:.4f} | Score {score:.4f}")
    return dice_final, acc, score

In [ ]:
def train_segmentation(model, train_loaders, val_loaders, epochs=10, lr=1e-3,
                       device=DEVICE, ckpt="best_model.pth"):
    model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    best = -1.0
    for ep in range(1, epochs + 1):
        model.train(); running = 0.0; nb = 0
        batches = list(iterate_mixed(train_loaders))      # 매 에폭 새로 셔플
        for x, y, cond in tqdm(batches, desc=f"Epoch {ep}/{epochs}", leave=False):
            x, y, cond = x.to(device), y.to(device), cond.to(device)
            optim.zero_grad()
            loss = seg_loss(model(x, cond), y)
            if torch.isnan(loss):
                continue
            loss.backward(); optim.step()
            running += loss.item(); nb += 1
        print(f"Epoch {ep}: avg train loss {running / max(nb, 1):.4f}")   # 진짜 평균
        print("  [val] ", end=""); _, _, score = evaluate(model, val_loaders, device=device)  # 테스트 아님
        if score > best:
            best = score; torch.save(model.state_dict(), ckpt)
            print(f"  ✅ best 갱신 (val {best:.4f}) → {ckpt}")
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"best 가중치 로드 (val {best:.4f})")
    return model

In [ ]:
model = UNetSegmentation(in_channels=16, cond_dim=COND_DIM).to(DEVICE)
if os.path.exists(BASELINE_MODEL):
    try:
        model.load_state_dict(torch.load(BASELINE_MODEL, map_location=DEVICE))
        print("baseline 가중치 로드됨")
    except Exception as e:
        print("baseline 구조 불일치(FiLM 추가) → 무작위 초기화:", repr(e)[:80])
else:
    print("baseline 가중치 없음 → 무작위 초기화로 학습합니다.")

In [ ]:
model = train_segmentation(model, train_loaders, val_loaders,
                           epochs=15, lr=1e-3, device=DEVICE)

In [ ]:
print("== 공개 테스트셋 성능 ==")
evaluate(model, test_loaders, device=DEVICE)

In [ ]:
import numpy as _np, zipfile as _zip
model.eval(); model.to(DEVICE)

@torch.no_grad()
def predict_block(ds, bs=16):
    loader = DataLoader(ds, batch_size=bs, shuffle=False)   # 순서 보존
    outs = []
    for x, y, cond in tqdm(loader, desc="제출 예측"):
        x, cond = x.to(DEVICE), cond.to(DEVICE)
        p = (torch.sigmoid(model(x, cond)) > 0.5).float()
        outs.append(p[:, 0].cpu().numpy().astype('uint8'))
    return _np.concatenate(outs, axis=0)

Y_pred_128 = predict_block(test_ds_128)
Y_pred_256 = predict_block(test_ds_256)
print('pred_a shapes:', Y_pred_128.shape, Y_pred_256.shape)
_np.savez('pred_a.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)
with _zip.ZipFile('submission.zip', 'w') as z:
    z.write('pred_a.npz')
print('✅ submission.zip 생성 완료 (pred_a.npz). Submissions 탭에 업로드하세요.')

In [ ]:
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치:", found)